# IslamicEval 2026 — Subtask 2: Hallucination Identification (RAG)

**Isnad AI** · end-to-end retrieval-augmented verification pipeline.

Last year (Subtask 1A) the job was to *detect* citation spans with a fine-tuned AraBERT.
This year the spans are **given** and the job is to *verify* each one — decide whether a
quoted **Ayah / matn / isnad / claimed_source** is `correct` or `incorrect`.

That reframing is why this notebook is built around **retrieval + matching (RAG)** against the
canonical corpora rather than a token classifier:

> A citation is `correct` when it *faithfully matches an authentic source*. So: retrieve the
> nearest canonical verse / hadith, measure how well the quoted span matches it, and threshold.

**Pipeline**

1. Load corpora — **prefers the cleaned CSVs from the preprocessing notebook** (`processed_dir`),
   falling back to raw `quranic_verses.json` / `six_hadith_books.json`
2. Multi-level Arabic normalization (diacritics → letters → optional morphology) — **identical
   تشكيل stripping to the preprocessing notebook**, so spans and corpus normalize the same way
3. Build retrieval indexes (char-n-gram TF-IDF for candidates + optional semantic embeddings)
4. Verify each segment type:
   - **Ayah / matn** → fuzzy + semantic similarity to nearest source, thresholded
   - **claimed_source** → parse the stated reference, compare to the matched source's true reference
   - **isnad** → compare chain to source (or documented fallback)
5. Tune thresholds on dev, write `submission.tsv`, score with the official metric.

Everything runs on a **≤13B / CPU-friendly** stack by default (no GPU required for the core
method), respecting the shared-task parameter limit.


## 0 · Setup

In [ ]:
# Core deps are light. rapidfuzz = fast fuzzy matching; scikit-learn = TF-IDF retrieval.
# sentence-transformers/faiss are OPTIONAL (semantic pass) — skip if you want CPU-only & fast.
!pip -q install rapidfuzz scikit-learn pandas numpy tqdm
# Optional semantic layer (comment out to stay ultra-light):
# !pip -q install sentence-transformers faiss-cpu
# Optional morphology (L4/L5 normalization):
# !pip -q install camel-tools
print("deps ready")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1 · Configuration

Point these at your files. Key names are auto-detected in the loaders, so you don't have to
rename anything. If a path is missing the notebook falls back to a small **synthetic demo** so
every cell still runs end-to-end.

In [ ]:
from pathlib import Path

CFG = {
    # ---- PREPROCESSED corpus from the preprocessing notebook (preferred source) ----
    # Point this at the same OUT_DIR you used there. The RAG loads the cleaned, تشكيل-free,
    # diacritic-augmented CSVs directly, so it never re-normalizes raw JSON and stays consistent.
    "processed_dir": "/content/drive/MyDrive/NAMAA Drive/shared_tasks/IslamicEval/Dataset/processed",

    # ---- RAW corpora (fallback only, if processed_dir is missing) ----
    "quran_path":  "/content/drive/MyDrive/NAMAA Drive/shared_tasks/IslamicEval/Dataset/quranic_verses.json",
    "hadith_path": "/content/drive/MyDrive/NAMAA Drive/shared_tasks/IslamicEval/Dataset/six_hadith_books.json",

    # ---- task data: responses + given segments (JSONL or JSON) ----
    #  expected per record: question / generated_answer / annotations[ {type, segments:[{segment_type,start,end,label?}]} ]
    "data_path":   "/content/drive/MyDrive/NAMAA Drive/shared_tasks/IslamicEval/Dataset/train.jsonl",

    # ---- retrieval / verification ----
    "topk": 15,                 # candidate shortlist size
    "norm_level": 3,            # 1=diacritics, 2=+letters, 3=+cleanup, 4=lemma, 5=root
    "use_semantic": False,      # set True to add embedding pass (needs sentence-transformers)
    "embed_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "cache_embeddings": True,   # save/reuse corpus embeddings in processed_dir (avoids recompute/OOM)

    # ---- thresholds (tuned later on dev; these are starting points) ----
    "tau_ayah": 0.90,
    "tau_matn": 0.82,
    "isnad_fallback": "correct",   # when isnad can't be grounded: "correct" (majority prior) or "incorrect"

    # ---- output ----
    "out_tsv": "/content/submission.tsv",
}
PROC = Path(CFG["processed_dir"])
print("processed_dir:", ("FOUND" if PROC.exists() else "MISSING"), CFG["processed_dir"])
for k in ("quran_path","hadith_path","data_path"):
    print(("FOUND " if Path(CFG[k]).exists() else "MISSING"), CFG[k])

## 2 · Arabic normalization (the single most important preprocessing step)

The same normalizer is applied to **both** the corpus and the quoted spans, so that an
undiacritized LLM quote can still match a fully-diacritized canonical verse. Levels are additive
(see the *Morphological Analysis* sheet in the companion workbook). L1–L3 are safe and high-win;
L4–L5 (lemma/root) are optional and should be A/B-tested on dev because they can over-merge
distinct verses.

In [ ]:
import re

# Full Quranic diacritics + annotation marks (matches the preprocessing notebook exactly), so a
# span is normalized identically to the corpus it is matched against. Covers tanwin, harakat,
# shadda, sukun, dagger alef, maddah, hamza marks and the Quranic annotation signs.
_TASHKEEL = re.compile(r'[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED]')
_TATWEEL  = '\u0640'
_NON_AR   = re.compile(r'[^\u0621-\u064A\s]')            # keep Arabic letters + whitespace
_SPACES   = re.compile(r'\s+')

def _letters(t):
    t = re.sub('[إأآٱ\u0671]', 'ا', t)                   # incl. alef-wasla
    t = t.replace('ى', 'ي').replace('ؤ', 'و').replace('ئ', 'ي')
    t = t.replace('ة', 'ه')      # ta-marbuta -> ha (aggressive but stabilizes matching)
    return t

_MORPH = None
def _get_morph():
    global _MORPH
    if _MORPH is None:
        from camel_tools.morphology.database import MorphologyDB
        from camel_tools.morphology.analyzer import Analyzer
        _MORPH = Analyzer(MorphologyDB.builtin_db(), 'NONE')
    return _MORPH

def normalize(text, level=3):
    if not text:
        return ""
    t = _TASHKEEL.sub('', str(text)).replace(_TATWEEL, '')      # L1
    if level >= 2:
        t = _letters(t)                                         # L2
    if level >= 3:
        t = _NON_AR.sub(' ', t)                                 # L3 cleanup
        t = _SPACES.sub(' ', t).strip()
    if level >= 4:                                              # L4 lemma (optional)
        an = _get_morph()
        out = []
        for w in t.split():
            a = an.analyze(w)
            out.append(a[0]['lex'] if a else w)
        t = _SPACES.sub(' ', ' '.join(out)).strip()
    return t

# quick check
for s in ["الرَّحْمَـٰنِ الرَّحِيمِ", "إِنَّآ أَعْطَيْنَاكَ"]:
    print(repr(s), '->', repr(normalize(s, 2)))

## 3 · Loaders (processed-Drive-first, key-name agnostic, synthetic fallback)

Load order for each corpus: **(1)** the cleaned CSVs written by the preprocessing notebook
(`processed_dir/quran_augmented.csv`, `hadith_augmented.csv`) — already تشكيل-free and
diacritic-augmented, so the RAG reuses exactly the same corpus as your other notebooks; **(2)** raw
JSON, normalized inline with the identical function; **(3)** a tiny synthetic sample so the
notebook always runs. The `text_norm` column from the processed files is used verbatim when present,
guaranteeing span↔corpus normalization parity.

In [ ]:
import json
import pandas as pd

def _read_json_any(path):
    '''Read .json (array) or .jsonl (one object per line).'''
    p = Path(path)
    if not p.exists():
        return None
    txt = p.read_text(encoding='utf-8').strip()
    if not txt:
        return []
    try:
        return json.loads(txt)                       # plain JSON array/object
    except json.JSONDecodeError:
        return [json.loads(ln) for ln in txt.splitlines() if ln.strip()]  # JSONL

def _first_key(d, keys):
    for k in keys:
        if k in d and d[k] not in (None, ""):
            return d[k]
    return None

def _cell(row, key):
    v = row[key] if key in row.index else None
    return None if (v is None or (isinstance(v, float) and pd.isna(v)) or v == "") else v

# ---------- processed-CSV loaders (preferred: reuse the cleaned Drive corpus) ----------
def _quran_from_processed():
    fp = PROC / "quran_augmented.csv"
    if not fp.exists():
        fp = PROC / "quran_clean.csv"
    if not fp.exists():
        return None
    df = pd.read_csv(fp).fillna("")
    verses = []
    for _, r in df.iterrows():
        txt = _cell(r, "text") or _cell(r, "text_raw")
        if not txt:
            continue
        verses.append({"text": txt,
                       "norm": _cell(r, "text_norm") or normalize(txt, CFG["norm_level"]),
                       "surah_id": _cell(r, "surah_id"),
                       "surah_name": _cell(r, "surah_name"),
                       "ayah_id": _cell(r, "ayah_id")})
    print(f"[quran] loaded {len(verses)} from processed: {fp.name}")
    return verses

def _hadith_from_processed():
    fp = PROC / "hadith_augmented.csv"
    if not fp.exists():
        fp = PROC / "hadith_clean.csv"
    if not fp.exists():
        return None
    df = pd.read_csv(fp).fillna("")
    hadiths = []
    for _, r in df.iterrows():
        txt = _cell(r, "text") or _cell(r, "text_raw")
        if not txt:
            continue
        isn = _cell(r, "isnad_raw") or _cell(r, "isnad") or ""
        hadiths.append({"text": txt,
                        "norm": _cell(r, "text_norm") or normalize(txt, CFG["norm_level"]),
                        "book": _cell(r, "book"), "book_id": _cell(r, "book_id"),
                        "isnad": isn, "isnad_norm": normalize(isn, CFG["norm_level"])})
    print(f"[hadith] loaded {len(hadiths)} from processed: {fp.name}")
    return hadiths

# ---------- raw-JSON loaders (fallback -> synthetic) ----------
def _quran_from_raw(path):
    data = _read_json_any(path)
    if not data:
        print("[quran] using synthetic sample")
        data = [
            {"surah_id":1,"surah_name":"الفاتحة","ayah_id":1,"ayah_text":"بِسْمِ اللَّهِ الرَّحْمَٰنِ الرَّحِيمِ"},
            {"surah_id":112,"surah_name":"الإخلاص","ayah_id":1,"ayah_text":"قُلْ هُوَ اللَّهُ أَحَدٌ"},
            {"surah_id":51,"surah_name":"الذاريات","ayah_id":56,"ayah_text":"وَمَا خَلَقْتُ الْجِنَّ وَالْإِنسَ إِلَّا لِيَعْبُدُونِ"},
        ]
    verses = []
    for d in data:
        text = _first_key(d, ["ayah_text","full_text","span_text","text"])
        if not text:
            continue
        verses.append({"text": text, "norm": normalize(text, CFG["norm_level"]),
                       "surah_id": _first_key(d, ["surah_id","surah","surahId"]),
                       "surah_name": _first_key(d, ["surah_name","surahName"]),
                       "ayah_id": _first_key(d, ["ayah_id","ayahId","verse_id","aya"])})
    print(f"[quran] {len(verses)} verses (raw)")
    return verses

def _hadith_from_raw(path):
    data = _read_json_any(path)
    if not data:
        print("[hadith] using synthetic sample")
        data = [
            {"hadithID":1,"title":"البخاري","Matn":"إنما الأعمال بالنيات وإنما لكل امرئ ما نوى",
             "isnad":"حدثنا الحميدي عبد الله بن الزبير عن سفيان عن يحيى بن سعيد"},
            {"hadithID":2,"title":"مسلم","Matn":"من حسن إسلام المرء تركه ما لا يعنيه","isnad":""},
        ]
    hadiths = []
    for d in data:
        matn = _first_key(d, ["Matn","matn","hadithTxt","hadith_text","text"])
        if not matn:
            continue
        hadiths.append({"text": matn, "norm": normalize(matn, CFG["norm_level"]),
                        "book": _first_key(d, ["title","book","BookName","collection"]),
                        "book_id": _first_key(d, ["BookID","book_id"]),
                        "isnad": _first_key(d, ["isnad","sanad","chain"]) or "",
                        "isnad_norm": normalize(_first_key(d, ["isnad","sanad","chain"]) or "", CFG["norm_level"])})
    print(f"[hadith] {len(hadiths)} matns (raw)")
    return hadiths

# ---------- dispatch: processed -> raw -> synthetic ----------
def load_quran():
    return _quran_from_processed() or _quran_from_raw(CFG["quran_path"])

def load_hadith():
    return _hadith_from_processed() or _hadith_from_raw(CFG["hadith_path"])

QURAN  = load_quran()
HADITH = load_hadith()

### 3b · Load the task responses + their given segments

Subtask 2 gives you the spans; you predict the label. This loader normalizes the official record
shape into a flat list of **segments to label**, recovering each span's text from the
character offsets in `generated_answer`. It also keeps the gold `label` when present (train/dev),
so we can tune thresholds and score offline.

In [ ]:
def load_segments(path):
    '''Flatten task records -> list of segments: {resp_id, ann_id, seg_type, span_text, gold?}.'''
    data = _read_json_any(path)
    if not data:
        print("[data] using synthetic demo (2 responses)")
        data = [
            {"id":"R000001",
             "generated_answer":"قال الله تعالى: قل هو الله احد. وهذا دليل على التوحيد.",
             "annotations":[{"type":"Ayah","segments":[
                 {"segment_type":"Ayah","start":16,"end":30,"label":"correct"},
                 {"segment_type":"claimed_source","start":0,"end":0,"label":"correct"}]}]},
            {"id":"R000002",
             "generated_answer":"روى البخاري: انما الاعمال بالخير وانما لكل امرئ ما نوى.",
             "annotations":[{"type":"Hadith","segments":[
                 {"segment_type":"matn","start":12,"end":52,"label":"incorrect"},
                 {"segment_type":"isnad","start":0,"end":0,"label":"N/A"}]}]},
        ]
    segs = []
    for rec in data:
        rid  = _first_key(rec, ["id","Response_ID","response_id","qid"])
        ans  = _first_key(rec, ["generated_answer","response","answer","Response","text"]) or ""
        anns = rec.get("annotations") or rec.get("citations") or []
        for ai, ann in enumerate(anns, 1):
            aid = _first_key(ann, ["annotation_id","id"]) or ai
            for s in (ann.get("segments") or [ann]):
                st = _first_key(s, ["segment_type","type","Segment_Type"])
                a  = _first_key(s, ["start","char_start","Span_Start","span_start"])
                b  = _first_key(s, ["end","char_end","Span_End","span_end"])
                # recover span text from offsets when available, else explicit span_text
                span_text = _first_key(s, ["span_text","text"])
                if span_text is None and a is not None and b is not None and int(b) > int(a):
                    span_text = ans[int(a):int(b)]
                segs.append({
                    "resp_id": rid, "ann_id": aid, "seg_type": st,
                    "span_text": span_text or "",
                    "gold": _first_key(s, ["label","Label","gold"]),   # may be None on test
                })
    print(f"[data] {len(segs)} segments across {len(data)} responses")
    return segs, data

SEGMENTS, RAW = load_segments(CFG["data_path"])
import pandas as pd
pd.DataFrame(SEGMENTS).head(8)

## 4 · Retrieval indexes

A char-n-gram TF-IDF index gives a fast, language-agnostic candidate shortlist (robust to Arabic
morphology because it works on sub-word character sequences). We then re-score the shortlist with
RapidFuzz for a precise similarity. An optional semantic pass (`use_semantic=True`) adds an
embedding retriever for paraphrase-tolerant recall.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

class Retriever:
    def __init__(self, records, ngram=(3,5)):
        self.records = records
        self.corpus  = [r["norm"] for r in records]
        self.vec = TfidfVectorizer(analyzer="char_wb", ngram_range=ngram, min_df=1)
        self.mat = self.vec.fit_transform(self.corpus) if self.corpus else None
        self._emb = None

    def build_embeddings(self, model_name, cache_tag=None):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer(model_name)
        cache = None
        if cache_tag and CFG.get("cache_embeddings"):
            # cache keyed by corpus size + model, saved in processed_dir so it survives restarts
            key = f"{cache_tag}_{len(self.records)}_{model_name.split('/')[-1]}.npy"
            cache = PROC / "emb_cache"; cache.mkdir(exist_ok=True)
            cache = cache / key
            if cache.exists():
                self._emb = np.load(cache)
                print(f"[emb] loaded cache {cache.name}")
                return
        self._emb = self.model.encode([r["text"] for r in self.records],
                                      convert_to_numpy=True, normalize_embeddings=True,
                                      show_progress_bar=True)
        if cache is not None:
            np.save(cache, self._emb); print(f"[emb] saved cache {cache.name}")

    def candidates(self, query_norm, k=15, semantic=False):
        idx = set()
        if self.mat is not None and query_norm:
            sims = linear_kernel(self.vec.transform([query_norm]), self.mat).ravel()
            idx.update(np.argsort(sims)[::-1][:k].tolist())
        if semantic and self._emb is not None:
            q = self.model.encode([query_norm], convert_to_numpy=True, normalize_embeddings=True)
            sims = (self._emb @ q[0])
            idx.update(np.argsort(sims)[::-1][:k].tolist())
        return [self.records[i] for i in idx]

print("Building Quran retriever...");  QRET = Retriever(QURAN)
print("Building Hadith retriever..."); HRET = Retriever(HADITH)
if CFG["use_semantic"]:
    QRET.build_embeddings(CFG["embed_model"], cache_tag="quran")
    HRET.build_embeddings(CFG["embed_model"], cache_tag="hadith")
print("indexes ready")

## 5 · Similarity scoring

For a quoted span we take the best of two RapidFuzz measures against each candidate:

- `token_set_ratio` — order-insensitive, forgiving of extra/missing words (good for full quotes),
- `partial_ratio` — best alignment of the span *inside* a longer verse (good for fragments).

The returned `best_score ∈ [0,1]` and the matched source record drive every downstream decision.

In [ ]:
from rapidfuzz import fuzz

def best_match(span_text, retriever, k, semantic):
    q = normalize(span_text, CFG["norm_level"])
    if not q:
        return 0.0, None
    cands = retriever.candidates(q, k=k, semantic=semantic)
    best, best_rec = 0.0, None
    for rec in cands:
        s = max(fuzz.token_set_ratio(q, rec["norm"]),
                fuzz.partial_ratio(q, rec["norm"])) / 100.0
        if s > best:
            best, best_rec = s, rec
    return best, best_rec

## 6 · `claimed_source` and `isnad` verifiers

**claimed_source** — parse the stated reference out of the response (a surah name + optional
verse number for Quran, or a collection name like البخاري / مسلم for Hadith) and compare it to the
*true* reference of the source that the Ayah/matn matched. This is scored only when the parent
text is correct, so we verify against the matched record.

**isnad** — genuinely the hardest and the biggest risk (it is 25% of the macro metric). If the
hadith corpus carries an isnad/sanad field we fuzzy-compare the quoted chain to it; otherwise we
fall back to the documented majority prior (`CFG['isnad_fallback']`) and flag it. Improving this
is the top lever for next iterations (see the Segment Strategy sheet).

In [ ]:
# surah-name -> id map, built straight from the corpus so it matches your file's spelling
SURAH_BY_NAME = {}
for v in QURAN:
    if v.get("surah_name") and v.get("surah_id") is not None:
        SURAH_BY_NAME[normalize(v["surah_name"], 2)] = v["surah_id"]

_ARABIC_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
def _find_number(text):
    m = re.search(r'\d+', text.translate(_ARABIC_DIGITS))
    return int(m.group()) if m else None

HADITH_BOOKS = ["البخاري","مسلم","الترمذي","النسائي","ابو داود","ابن ماجه","احمد","مالك","الدارمي"]

def verify_claimed_source(span_text, matched_rec, kind):
    '''kind = 'Ayah' or 'matn'. Returns 'correct'/'incorrect'.'''
    claim = normalize(span_text, 2)
    if matched_rec is None or not claim:
        return "incorrect"
    if kind == "Ayah":
        # does the claim name the same surah (and verse if given) as the matched verse?
        claimed_surah = next((sid for name, sid in SURAH_BY_NAME.items() if name and name in claim), None)
        if claimed_surah is None:
            return "incorrect"
        if str(claimed_surah) != str(matched_rec.get("surah_id")):
            return "incorrect"
        n = _find_number(span_text)
        if n is not None and matched_rec.get("ayah_id") is not None:
            return "correct" if str(n) == str(matched_rec.get("ayah_id")) else "incorrect"
        return "correct"
    else:  # hadith collection attribution
        claimed_book = next((b for b in HADITH_BOOKS if normalize(b,2) in claim), None)
        true_book = normalize(str(matched_rec.get("book") or ""), 2)
        if claimed_book is None:
            return "incorrect"
        return "correct" if normalize(claimed_book,2) in true_book or true_book in normalize(claimed_book,2) else "incorrect"

def verify_isnad(span_text, matched_rec):
    q = normalize(span_text, CFG["norm_level"])
    src = (matched_rec or {}).get("isnad_norm") or ""
    if not q:
        return CFG["isnad_fallback"]
    if src:                                   # grounded comparison possible
        s = max(fuzz.token_set_ratio(q, src), fuzz.partial_ratio(q, src)) / 100.0
        return "correct" if s >= 0.75 else "incorrect"
    return CFG["isnad_fallback"]              # documented fallback

## 7 · Label one segment

Ties the pieces together. For `Ayah`/`matn` we retrieve → score → threshold. For
`claimed_source`/`isnad` we first find the parent text's best source match, then run the
structured verifier. The matched score is kept for inspection/tuning.

In [ ]:
def label_segment(seg, tau_ayah, tau_matn, semantic):
    st = (seg["seg_type"] or "").strip()
    txt = seg["span_text"]

    if st == "Ayah":
        score, rec = best_match(txt, QRET, CFG["topk"], semantic)
        return ("correct" if score >= tau_ayah else "incorrect"), score, rec
    if st == "matn":
        score, rec = best_match(txt, HRET, CFG["topk"], semantic)
        return ("correct" if score >= tau_matn else "incorrect"), score, rec
    if st == "claimed_source":
        # match against BOTH corpora, keep whichever is closer, then check the reference
        sa, ra = best_match(txt, QRET, CFG["topk"], semantic)
        sh, rh = best_match(txt, HRET, CFG["topk"], semantic)
        if sa >= sh:
            return verify_claimed_source(txt, ra, "Ayah"), sa, ra
        return verify_claimed_source(txt, rh, "matn"), sh, rh
    if st == "isnad":
        sh, rh = best_match(txt, HRET, CFG["topk"], semantic)
        return verify_isnad(txt, rh), sh, rh
    # unknown type -> safe default
    return "incorrect", 0.0, None

## 8 · Run over all segments

In [ ]:
from tqdm.auto import tqdm

def run(segments, tau_ayah=None, tau_matn=None, semantic=None):
    tau_ayah = CFG["tau_ayah"] if tau_ayah is None else tau_ayah
    tau_matn = CFG["tau_matn"] if tau_matn is None else tau_matn
    semantic = CFG["use_semantic"] if semantic is None else semantic
    out = []
    for seg in tqdm(segments):
        label, score, rec = label_segment(seg, tau_ayah, tau_matn, semantic)
        out.append({**seg, "pred": label, "score": round(score, 3),
                    "matched": (rec or {}).get("text", "")[:60]})
    return pd.DataFrame(out)

pred_df = run(SEGMENTS)
pred_df.head(10)

## 9 · Offline evaluation & threshold tuning (when gold is present)

The official metric is **macro accuracy over the 4 segment types, excluding gold `N/A`**. This
cell reproduces it, then sweeps the Ayah/matn thresholds to pick the pair that maximizes macro
accuracy on your labelled split. Log the winning config in the workbook's *Experiments Log*.

In [ ]:
SEG_TYPES = ["Ayah", "matn", "isnad", "claimed_source"]

def macro_accuracy(df):
    per = {}
    for st in SEG_TYPES:
        sub = df[(df["seg_type"] == st) & (df["gold"].isin(["correct","incorrect"]))]
        per[st] = (sub["pred"] == sub["gold"]).mean() if len(sub) else float("nan")
    valid = [v for v in per.values() if v == v]
    per["MACRO"] = sum(valid)/len(valid) if valid else float("nan")
    return per

has_gold = any(s["gold"] in ("correct","incorrect") for s in SEGMENTS)
if has_gold:
    print("Current config:", macro_accuracy(pred_df))

    best, best_cfg = -1, None
    for ta in [round(x,2) for x in np.arange(0.80, 0.99, 0.02)]:
        for tm in [round(x,2) for x in np.arange(0.70, 0.95, 0.02)]:
            m = macro_accuracy(run(SEGMENTS, tau_ayah=ta, tau_matn=tm, semantic=False))["MACRO"]
            if m == m and m > best:
                best, best_cfg = m, (ta, tm)
    print(f"\nBEST macro acc {best:.3f} at tau_ayah={best_cfg[0]}, tau_matn={best_cfg[1]}")
    CFG["tau_ayah"], CFG["tau_matn"] = best_cfg
    pred_df = run(SEGMENTS)
    print("Tuned per-type:", macro_accuracy(pred_df))
else:
    print("No gold labels in this split (test set) — skipping tuning. "
          "Use your dev split to tune, then apply the same thresholds here.")

## 10 · Write the submission

TSV with `Response_ID, Annotation_ID, Segment_Type, Label`. Per the rules we **do not** emit
`N/A` rows and **do not** emit rows for no-citation responses; the scorer excludes gold-`N/A`
automatically.

In [ ]:
sub = pred_df[["resp_id","ann_id","seg_type","pred"]].copy()
sub.columns = ["Response_ID","Annotation_ID","Segment_Type","Label"]
sub = sub[sub["Label"].isin(["correct","incorrect"])]      # never submit N/A
sub.to_csv(CFG["out_tsv"], sep="\t", index=False)
print(f"wrote {len(sub)} rows -> {CFG['out_tsv']}")
sub.head()

In [ ]:
# zip for upload (mirrors your 2025 submission workflow)
import zipfile, os
zip_path = "/content/submission.zip"
with zipfile.ZipFile(zip_path, "w") as zf:
    zf.write(CFG["out_tsv"], os.path.basename(CFG["out_tsv"]))
print("zipped ->", zip_path)

## 11 · (Optional) run the official scorer locally

If you have `task2_scoring.py` and the gold TSV, drop them in the folder layout the organizers
expect and run it — this is the ground truth for your dev numbers.

In [ ]:
# import os
# ROOT = "/content/scoring"
# os.makedirs(f"{ROOT}/input/ref", exist_ok=True)
# os.makedirs(f"{ROOT}/input/res", exist_ok=True)
# os.makedirs(f"{ROOT}/output", exist_ok=True)
# !cp "{CFG['out_tsv']}" "{ROOT}/input/res/"
# !cp "/content/drive/MyDrive/.../gold_subtask2.tsv" "{ROOT}/input/ref/"
# %env SCORING_ROOT={ROOT}
# !python task2_scoring.py
# import json; print(json.load(open(f"{ROOT}/output/scores.json")))

## 12 · Where to push next

- **isnad (biggest lever, 25% of macro):** get a hadith source that actually carries the chain
  (or a narrator DB) so `verify_isnad` is grounded instead of falling back to a prior.
- **matn recall:** add `nine_hadith_books.csv` to the hadith retriever to cover cross-collection
  wording variants; turn on `use_semantic=True`.
- **Morphology (L4/L5):** switch `norm_level` to 4 and A/B-test on dev — log both in the workbook.
- **Supervised verifier (M5):** if fuzzy+semantic plateaus, fine-tune AraBERT on
  `(span, retrieved_source) → correct/incorrect` pairs, reusing your 2025 training stack.
- **Correction (Subtask 3):** the matched source record already *is* the correction — emit
  `matched_rec['text']` for spans you label `incorrect`, or `خطأ` when `best_score` is very low.
